
# Temporal Graph Learning for Ethereum Fraud Detection

This notebook provides an end-to-end workflow to build a dynamic transaction graph from the
`ETHdata` and `Sub-CATS-Txs` folders and to train graph neural network (GNN) models that flag
potentially fraudulent Ethereum wallets. It is designed to run directly on Google Colab with GPU
support. The pipeline covers:

* Loading hundreds of thousands of JSON transaction files from Google Drive in a streaming fashion.
* Building a temporal, directed, weighted multigraph where nodes represent wallets and edges represent
  on-chain transfers (internal, ERC-20/721/1155, and normal transactions).
* Engineering structural features such as degree centrality, clustering coefficient, PageRank, and
  transaction statistics that enrich node representations.
* Training both a static GraphSAGE classifier and a temporal graph neural network (GCLSTM) to predict
  whether a wallet belongs to a blacklist.
* Visualizing suspicious subgraphs to aid investigations.

> **Tip:** The raw dataset is very large (~80k folders). Use the sampling parameters provided in the
> loading functions to iterate during experimentation and gradually scale up once the pipeline works
> end-to-end.



## 1. Environment setup

Make sure the runtime uses a GPU (`Runtime ➜ Change runtime type ➜ GPU`). The following cell installs
PyTorch (CUDA build), PyTorch Geometric, and additional utilities needed for temporal graph learning
and visualization.


In [ ]:

%%capture
!pip install -q torch==2.2.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv     -f https://data.pyg.org/whl/torch-2.2.1+cu121.html
!pip install -q torch-geometric torch-geometric-temporal networkx pandas numpy tqdm seaborn scikit-learn pyvis plotly


In [ ]:

import os
import json
from pathlib import Path
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv
from torch_geometric_temporal.signal import DynamicGraphTemporalSignal
from torch_geometric_temporal.nn.recurrent import GCLSTM
import networkx as nx
from tqdm.notebook import tqdm



## 2. Mount Google Drive and configure dataset paths

Upload the `real-CATs` folder (containing both `ETHdata` and `Sub-CATS-Txs`) to your Google Drive.
The following cell mounts Drive and defines helper paths. Update `BASE_DATA_DIR` if your directory
structure differs.


In [ ]:

from google.colab import drive

drive.mount('/content/drive')
BASE_DATA_DIR = Path('/content/drive/MyDrive/real-CATs')
ETHDATA_DIR = BASE_DATA_DIR / 'ETHdata'
SUBCATS_DIR = BASE_DATA_DIR / 'Sub-CATS-Txs'
assert ETHDATA_DIR.exists(), f"Missing ETHdata directory at {ETHDATA_DIR}"
assert SUBCATS_DIR.exists(), f"Missing Sub-CATS-Txs directory at {SUBCATS_DIR}"



## 3. Data ingestion helpers

Each wallet folder may contain `normal_transactions.json`, `internal_transactions.json`,
`ERC_20_transactions.json`, `ERC_721_transactions.json`, and `ERC_1155_transactions.json` files.
The utilities below parse these JSON files lazily so that only a subset needs to be loaded in memory
at any time. Use the `max_wallets` and `max_transactions_per_wallet` arguments to throttle loading
while prototyping.


In [ ]:

TRANSACTION_FILES = {
    'normal_transactions.json': 'normal',
    'internal_transactions.json': 'internal',
    'ERC_20_transactions.json': 'erc20',
    'ERC_721_transactions.json': 'erc721',
    'ERC_1155_transactions.json': 'erc1155',
}


def iter_wallet_directories(max_wallets: Optional[int] = None) -> Iterable[Path]:
    '''Yield wallet directories from both ETHdata and Sub-CATS-Txs.'''
    roots = [ETHDATA_DIR, SUBCATS_DIR]
    count = 0
    for root in roots:
        for wallet_dir in root.iterdir():
            if not wallet_dir.is_dir():
                continue
            yield wallet_dir
            count += 1
            if max_wallets is not None and count >= max_wallets:
                return


def load_wallet_transactions(wallet_dir: Path, max_transactions_per_wallet: Optional[int] = None) -> List[Dict]:
    '''Parse all JSON transaction files in a wallet directory.'''
    transactions: List[Dict] = []
    for file_name, tx_type in TRANSACTION_FILES.items():
        file_path = wallet_dir / file_name
        if not file_path.exists():
            continue
        with open(file_path, 'r') as f:
            raw_items = json.load(f)
        for item in raw_items:
            record = {
                'from': item.get('from'),
                'to': item.get('to'),
                'value': float(item.get('value', 0)) / 1e18,
                'hash': item.get('hash'),
                'timeStamp': int(item.get('timeStamp')),
                'blockNumber': int(item.get('blockNumber')),
                'gas': float(item.get('gas', 0)),
                'gasUsed': float(item.get('gasUsed', 0)),
                'isError': int(item.get('isError', 0)),
                'type': tx_type,
                'wallet_dir': wallet_dir.name,
            }
            if record['from'] is None or record['to'] is None:
                continue
            transactions.append(record)
            if max_transactions_per_wallet is not None and len(transactions) >= max_transactions_per_wallet:
                break
        if max_transactions_per_wallet is not None and len(transactions) >= max_transactions_per_wallet:
            break
    return transactions


def load_transaction_dataframe(max_wallets: Optional[int] = 2000,
                               max_transactions_per_wallet: Optional[int] = None) -> pd.DataFrame:
    '''Aggregate transactions from the dataset into a single DataFrame.'''
    rows: List[Dict] = []
    for wallet_dir in tqdm(iter_wallet_directories(max_wallets=max_wallets), desc='Loading wallets'):
        rows.extend(load_wallet_transactions(wallet_dir, max_transactions_per_wallet))
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError('No transactions were loaded. Check dataset paths or sampling limits.')
    df['timestamp'] = pd.to_datetime(df['timeStamp'], unit='s', utc=True)
    df = df.sort_values('timestamp').reset_index(drop=True)
    return df



### Load a manageable transaction sample

The parameters below keep the initial sample small (e.g., ~5k wallets). Increase them gradually as the
pipeline stabilizes. The resulting DataFrame includes all directed edges (source wallet ➜ destination
wallet) along with metadata.


In [ ]:

MAX_WALLETS = 5000          # Increase cautiously; 0 or None loads the full dataset
MAX_TX_PER_WALLET = 500     # Cap to avoid extremely large hot wallets during prototyping

tx_df = load_transaction_dataframe(max_wallets=MAX_WALLETS,
                                    max_transactions_per_wallet=MAX_TX_PER_WALLET)
tx_df.head()



## 4. Node dictionary and base features

We first build an integer index for each wallet address and compute core statistics such as transaction
counts, transferred value, error rates, and token distribution. Wallets with no transactions are
removed automatically.


In [ ]:

addresses = pd.Index(pd.unique(tx_df[['from', 'to']].values.ravel()))
address_to_id = {addr: idx for idx, addr in enumerate(addresses)}

tx_df['src_id'] = tx_df['from'].map(address_to_id)
tx_df['dst_id'] = tx_df['to'].map(address_to_id)

feature_df = (
    tx_df.groupby('from').agg(
        out_tx_count=('hash', 'count'),
        out_value_total=('value', 'sum'),
        out_value_mean=('value', 'mean'),
        out_is_error_rate=('isError', 'mean'),
        out_internal_ratio=('type', lambda x: np.mean(np.array(x == 'internal', dtype=float))),
    )
    .join(
        tx_df.groupby('to').agg(
            in_tx_count=('hash', 'count'),
            in_value_total=('value', 'sum'),
            in_value_mean=('value', 'mean'),
        ),
        how='outer'
    )
    .fillna(0.0)
)

feature_df['total_tx'] = feature_df['in_tx_count'] + feature_df['out_tx_count']
feature_df['net_value'] = feature_df['in_value_total'] - feature_df['out_value_total']
feature_df = feature_df.reindex(addresses, fill_value=0.0)
feature_df.head()



## 5. Advanced graph-structural features

To capture a wallet's role within the broader network we compute graph metrics on an undirected view
of the transaction network. These include degree centrality, clustering coefficient, PageRank, and
betweenness. Use the `centrality_sample` parameter to limit the graph before the expensive NetworkX
calculations.


In [ ]:

CENTRALITY_SAMPLE = 30000  # Number of edges to keep when computing NetworkX measures

sampled_edges = tx_df[['src_id', 'dst_id', 'value']].head(CENTRALITY_SAMPLE)
G = nx.DiGraph()
G.add_nodes_from(range(len(addresses)))
for row in sampled_edges.itertuples(index=False):
    G.add_edge(int(row.src_id), int(row.dst_id), weight=float(row.value))

G_undirected = G.to_undirected()

degree_centrality = nx.degree_centrality(G)
in_degree_centrality = nx.in_degree_centrality(G)
out_degree_centrality = nx.out_degree_centrality(G)
clustering = nx.clustering(G_undirected)
pagerank = nx.pagerank(G, weight='weight')
betweenness = nx.betweenness_centrality(G, k=min(500, G.number_of_nodes()), seed=42)

centrality_df = pd.DataFrame({
    'degree_centrality': pd.Series(degree_centrality),
    'in_degree_centrality': pd.Series(in_degree_centrality),
    'out_degree_centrality': pd.Series(out_degree_centrality),
    'clustering_coeff': pd.Series(clustering),
    'pagerank': pd.Series(pagerank),
    'betweenness': pd.Series(betweenness),
}).fillna(0.0)

centrality_df = centrality_df.reindex(range(len(addresses)), fill_value=0.0)
centrality_df.head()



## 6. Combine features and scale

We merge the statistical and centrality features, standardize them, and create tensors for PyTorch
Geometric. The resulting `x` matrix will be shared by both the static and temporal models.


In [ ]:

node_features = feature_df.join(centrality_df)
feature_columns = node_features.columns
scaler = StandardScaler()
node_features_scaled = scaler.fit_transform(node_features.values)

x = torch.tensor(node_features_scaled, dtype=torch.float32)
edge_index = torch.tensor(tx_df[['src_id', 'dst_id']].values.T, dtype=torch.long)
edge_weight = torch.tensor(tx_df['value'].values, dtype=torch.float32)
print(f"Number of wallets: {x.size(0)}")
print(f"Number of transactions (edges): {edge_index.size(1)}")



## 7. Load blacklist labels

If you have curated labels (e.g., Chainalysis, CipherTrace, or community-reported scams), place a CSV
file under `real-CATs/labels/address_labels.csv` with columns `address` and `label` (1 for fraudulent,
0 for benign). The fallback below fabricates labels based on high out-degree wallets so that the rest
of the notebook remains executable.


In [ ]:

LABEL_PATH = BASE_DATA_DIR / 'labels' / 'address_labels.csv'
if LABEL_PATH.exists():
    label_df = pd.read_csv(LABEL_PATH)
    label_df['address'] = label_df['address'].str.lower()
    label_df = label_df[label_df['address'].isin(addresses)]
    labels = pd.Series(0, index=addresses)
    labels.loc[label_df['address']] = label_df['label'].astype(int).values
else:
    print('Warning: No external labels found. Generating heuristic labels for demonstration.')
    labels = pd.Series(0, index=addresses)
    suspicious_mask = (
        feature_df['out_tx_count'] > feature_df['out_tx_count'].quantile(0.99)
    ) | (
        centrality_df['pagerank'] > centrality_df['pagerank'].quantile(0.99)
    )
    labels.loc[suspicious_mask[suspicious_mask].index] = 1

y = torch.tensor(labels.values, dtype=torch.long)
num_fraud = int(labels.sum())
print(f"Fraudulent wallets in sample: {num_fraud}")



## 8. Train a static GraphSAGE classifier

We perform an 80/20 train-test split across labeled wallets and train a two-layer GraphSAGE model.
Neighbor sampling is used to scale mini-batch training on large graphs.


In [ ]:

SEED = 42
torch.manual_seed(SEED)

labeled_indices = torch.nonzero(y >= 0, as_tuple=False).view(-1)
train_idx, test_idx = train_test_split(labeled_indices.numpy(), test_size=0.2, random_state=SEED,
                                       stratify=y[labeled_indices].numpy())
train_idx = torch.tensor(train_idx, dtype=torch.long)
test_idx = torch.tensor(test_idx, dtype=torch.long)

class GraphSAGEClassifier(torch.nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int = 2, dropout: float = 0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = torch.dropout(x, p=self.dropout, train=self.training)
        x = self.conv2(x, edge_index)
        x = torch.relu(x)
        x = torch.dropout(x, p=self.dropout, train=self.training)
        x = self.lin(x)
        return x

model = GraphSAGEClassifier(in_channels=x.size(1), hidden_channels=128)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()

data = Data(x=x, edge_index=edge_index, y=y)
train_loader = NeighborLoader(data,
                              num_neighbors=[20, 10],
                              input_nodes=train_idx,
                              batch_size=2048,
                              shuffle=True)


def evaluate(model: torch.nn.Module, indices: torch.Tensor) -> Dict[str, float]:
    model.eval()
    with torch.no_grad():
        logits = model(x, edge_index)
        preds = logits.argmax(dim=-1)[indices]
        probas = torch.softmax(logits, dim=-1)[indices, 1]
        report = classification_report(y[indices].cpu().numpy(), preds.cpu().numpy(), output_dict=True, zero_division=0)
        auc = roc_auc_score(y[indices].cpu().numpy(), probas.cpu().numpy())
    return {
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc': auc,
    }

EPOCHS = 15
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.item())
    metrics = evaluate(model, test_idx)
    print(f"Epoch {epoch:02d} | Loss {epoch_loss:.4f} | Precision {metrics['precision']:.3f} | "
          f"Recall {metrics['recall']:.3f} | F1 {metrics['f1']:.3f} | AUC {metrics['auc']:.3f}")



## 9. Build a temporal graph signal

We bucket transactions by week to create a sequence of graph snapshots. Each snapshot reuses the same
node feature matrix but updates edge connectivity and weights. Targets remain the same across time.


In [ ]:

WEEKLY_BIN = '7D'

bucketed = tx_df.copy()
bucketed['time_bin'] = bucketed['timestamp'].dt.to_period(WEEKLY_BIN).dt.to_timestamp()
edge_indices = []
edge_weights = []
features_sequence = []
targets_sequence = []
window_timestamps = []

for time_bin, group in bucketed.groupby('time_bin'):
    if group.empty:
        continue
    edge_indices.append(group[['src_id', 'dst_id']].to_numpy().T)
    edge_weights.append(group['value'].to_numpy())
    features_sequence.append(node_features_scaled)
    targets_sequence.append(labels.values)
    window_timestamps.append(time_bin)

print(f"Temporal windows: {len(edge_indices)}")

temporal_signal = DynamicGraphTemporalSignal(edge_indices=edge_indices,
                                             edge_weights=edge_weights,
                                             features=features_sequence,
                                             targets=targets_sequence)



## 10. Temporal GNN (GCLSTM)

We train a Gated Graph Convolutional LSTM over the temporal sequence. The model processes each weekly
snapshot and accumulates hidden states. Performance is evaluated on the last 20% of time windows.


In [ ]:

class TemporalClassifier(torch.nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int = 2):
        super().__init__()
        self.recurrent = GCLSTM(in_channels, hidden_channels)
        self.linear = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor) -> torch.Tensor:
        h, _ = self.recurrent(x, edge_index, edge_weight)
        out = self.linear(h)
        return out

temporal_model = TemporalClassifier(in_channels=x.size(1), hidden_channels=128)
temporal_optimizer = torch.optim.Adam(temporal_model.parameters(), lr=1e-3, weight_decay=1e-4)
temporal_criterion = torch.nn.CrossEntropyLoss()

num_windows = len(edge_indices)
train_cut = int(num_windows * 0.8)

for epoch in range(1, 8):
    temporal_model.train()
    total_loss = 0.0
    temporal_model.recurrent.reset_parameters()
    for snapshot_index, snapshot in enumerate(temporal_signal):
        edge_idx = torch.tensor(snapshot.edge_index, dtype=torch.long)
        edge_w = torch.tensor(snapshot.edge_weight, dtype=torch.float32)
        features = torch.tensor(snapshot.x, dtype=torch.float32)
        targets = torch.tensor(snapshot.y, dtype=torch.long)

        logits = temporal_model(features, edge_idx, edge_w)
        loss = temporal_criterion(logits, targets)
        temporal_optimizer.zero_grad()
        loss.backward()
        temporal_optimizer.step()

        if snapshot_index < train_cut:
            total_loss += float(loss.item())

    temporal_model.eval()
    preds_all = []
    targets_all = []
    probas_all = []
    with torch.no_grad():
        for snapshot_index, snapshot in enumerate(temporal_signal):
            if snapshot_index < train_cut:
                continue
            edge_idx = torch.tensor(snapshot.edge_index, dtype=torch.long)
            edge_w = torch.tensor(snapshot.edge_weight, dtype=torch.float32)
            features = torch.tensor(snapshot.x, dtype=torch.float32)
            targets = torch.tensor(snapshot.y, dtype=torch.long)
            logits = temporal_model(features, edge_idx, edge_w)
            probas = torch.softmax(logits, dim=-1)[:, 1]
            preds = logits.argmax(dim=-1)
            preds_all.extend(preds.cpu().numpy())
            targets_all.extend(targets.cpu().numpy())
            probas_all.extend(probas.cpu().numpy())
    report = classification_report(targets_all, preds_all, output_dict=True, zero_division=0)
    auc = roc_auc_score(targets_all, probas_all)
    print(f"Epoch {epoch:02d} | Train loss {total_loss:.4f} | Temporal Precision {report['1']['precision']:.3f} | "
          f"Recall {report['1']['recall']:.3f} | F1 {report['1']['f1-score']:.3f} | AUC {auc:.3f}")



## 11. Visualize suspicious clusters

The PyVis visualization below highlights a subgraph centered around the highest PageRank wallets
flagged as fraudulent. Customize `TOP_K` to inspect more addresses. The resulting HTML file can be
opened directly in Colab.


In [ ]:

from pyvis.network import Network

def visualize_suspicious_subgraph(top_k: int = 50, output_html: str = 'suspicious_wallets.html') -> str:
    top_suspicious = labels.sort_values(ascending=False).head(top_k).index
    mask = tx_df['from'].isin(top_suspicious) | tx_df['to'].isin(top_suspicious)
    subgraph_df = tx_df[mask].copy()

    net = Network(height='750px', width='100%', bgcolor='#0b0d17', font_color='white', directed=True)
    for addr in top_suspicious:
        node_id = address_to_id[addr]
        node_label = 'Fraudulent' if labels.loc[addr] == 1 else 'Benign'
        score = centrality_df.iloc[node_id]['pagerank']
        color = '#ff4d4d' if labels.loc[addr] == 1 else '#4d79ff'
        net.add_node(addr, label=f"{addr[:10]}...", title=f"{node_label}<br>PageRank: {score:.4e}", color=color)

    for row in subgraph_df.itertuples(index=False):
        net.add_edge(row.from_, row.to, title=f"Value: {row.value:.4f} ETH<br>Type: {row.type}")

    net.show(output_html)
    return output_html

html_path = visualize_suspicious_subgraph(top_k=50)
print(f"Interactive visualization saved to {html_path}")



## 12. Next steps

* Replace heuristic labels with verified blacklists for reliable evaluation.
* Tune sampling parameters, window granularity, and model hyperparameters to improve recall.
* Experiment with more advanced temporal GNNs (e.g., TGAT, TGN) once the baseline pipeline is stable.
* Export trained embeddings and share them with downstream alerting systems or dashboards.
